# Adaptive flux driver validation

Cross-checks of `total_fluxes` and `sum_fluxes` against brute-force
oversized-window sums and the canonical-half doubling identity, on all four
orbit classes. All gates asserted so re-execution is a regression test.

In [1]:
import numpy as np
from hblteuk.geo import MiniGeo
from hblteuk.jax import total_fluxes, solve_modes, sum_fluxes


def brute_total(orbit, lmax, k_half, n_half, **kw):
    modes = [(ll, mm, kk, nn) for ll in range(2, lmax + 1)
             for mm in range(0, ll + 1)
             for kk in range(-k_half, k_half + 1)
             for nn in range(-n_half, n_half + 1)
             if (mm > 0) or (mm == 0 and kk > 0)
             or (mm == 0 and kk == 0 and nn > 0)]
    arr = np.array(modes)
    sol = solve_modes(orbit, -2, arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3],
                      return_radial=False, **kw)
    return sum_fluxes(sol, orbit)


# Per-case tol AND brute-window padding. The generic (2D) case uses a
# milder orbit and tight padding: its brute-force cross-check re-solves an
# even larger window in one shot, and a strong-field 2D window runs to
# 10^5 modes -- too heavy for an example. (Strong-field generic amplitudes
# are validated per mode against pybhpt in tests/test_jax_generic.py; here
# we only cross-check the driver's summation on a tractable window.)
# columns: name, orbit, tol, (k_pad, n_pad), solver kwargs
cases = [("eccentric", MiniGeo(0.9, 10.0, 0.2, 1.0), 1e-5, (0, 4),
          dict(nsamples=2**10)),
         ("spherical", MiniGeo(0.9, 10.0, 0.0, 0.5), 1e-5, (3, 0),
          dict(ntheta=2**8)),
         ("generic",   MiniGeo(0.5, 12.0, 0.15, 0.8), 1e-4, (1, 2),
          dict(nsamples=2**9, ntheta=2**7))]
for name, orb, tol, (kpad, npad), kw in cases:
    got = total_fluxes(orb, tol=tol)
    lmax = int(got.table["l"].max()) + 2
    # degenerate axes stay collapsed (equatorial -> no k, circular -> no n)
    kh = 0 if abs(orb.x) == 1 else int(np.abs(got.table["k"]).max()) + kpad
    nh = 0 if orb.e == 0 else int(np.abs(got.table["n"]).max()) + npad
    ref = brute_total(orb, lmax, kh, nh, **kw)
    rE = abs(got.Edot[0] / ref.Edot[0] - 1)
    rL = abs(got.Ldot[0] / ref.Ldot[0] - 1)
    print(f"{name:10s}: dEdot {rE:.2e}, dLdot {rL:.2e}")
    assert rE < 5 * tol and rL < 5 * tol

eccentric : dEdot 1.64e-07, dLdot 1.39e-07


spherical : dEdot 3.13e-07, dLdot 1.43e-07


generic   : dEdot 1.39e-06, dLdot 1.12e-06


In [2]:
orb = MiniGeo(0.9, 8.0, 0.3, 0.6)
modes = [(ll, mm, kk, nn) for ll in (2, 3) for mm in (1, 2)
         for kk in (-1, 0, 1) for nn in (-1, 0, 1)]
arr = np.array(modes)
half = solve_modes(orb, -2, arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3],
                   nsamples=2**8, ntheta=2**6)
got = sum_fluxes(half, orb)
from hblteuk.jax import flux_modes
l, m, k, n = half.l, half.m, half.k, half.n
full = solve_modes(orb, -2, np.concatenate([l, l]), np.concatenate([m, -m]),
                   np.concatenate([k, -k]), np.concatenate([n, -n]),
                   nsamples=2**8, ntheta=2**6)
fx = flux_modes(full.amplitudes, full.m, full.omega, full.eigenvalues, orb.a,
                k=full.k, n=full.n, carter_frequencies=orb.carter_frequencies())
for tot, F, name in [(got.Edot, fx.Edot, "Edot"), (got.Ldot, fx.Ldot, "Ldot"),
                     (got.Qdot, fx.Qdot, "Qdot")]:
    rel = np.max(np.abs(tot / np.asarray(F).sum(axis=0) - 1))
    print(f"{name}: doubling vs full plane {rel:.2e}")
    assert rel < 1e-7

Edot: doubling vs full plane 5.55e-16
Ldot: doubling vs full plane 5.55e-16
Qdot: doubling vs full plane 1.22e-15


In [3]:
import warnings
orb = MiniGeo(0.9, 10.0, 0.2, 1.0)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    capped = total_fluxes(orb, tol=1e-12, lmax_cap=4)
assert any("lmax_cap" in str(x.message) for x in w)
assert not all(capped.converged.values())
print("cap path: warned and flagged converged =", capped.converged)

# two independent error sources, reported separately:
#  - error_bound: numerical (amplitude-precision) error of the modes we DID
#    compute, propagated from the per-mode fft estimates
#  - truncation error: from stopping the mode sum, governed by edge_report
#    (edge fraction x total), NOT by error_bound
got = total_fluxes(orb, tol=1e-4)
ref = brute_total(orb, int(got.table["l"].max()) + 3, 0,
                  int(np.abs(got.table["n"]).max()) + 5, nsamples=2**10)
trunc_err = abs(got.Edot[0] - ref.Edot[0])
edge_est = max(got.edge_report[a][0] for a in got.edge_report) * got.Edot[0]
print(f"Edot amplitude-precision bound: {got.error_bound[0,0]:.3e}")
print(f"Edot truncation error: true {trunc_err:.3e}, "
      f"edge_report estimate {edge_est:.3e}")
assert got.error_bound[0, 0] >= 0
# the edge_report-based estimate brackets the true truncation error
assert edge_est >= trunc_err

cap path: warned and flagged converged = {'Edot': False, 'Ldot': False, 'Qdot': False}


Edot amplitude-precision bound: 1.661e-18
Edot truncation error: true 6.715e-10, edge_report estimate 2.637e-09
